In [5]:
import urllib.request
from pathlib import Path

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
enron_file = data_dir / 'enron_spam_data.zip'
enron_url = 'https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip'

if not enron_file.exists():
    print("Downloading Enron spam dataset...")
    urllib.request.urlretrieve(enron_url, enron_file)
    print("Download completed successfully.")
else:
    print("Dataset already exists.")

Dataset already exists.


In [6]:
from sklearn.model_selection import train_test_split
import pandas as pd 

raw_data = pd.read_csv(enron_file, compression='zip')
df = pd.DataFrame({
    'text': raw_data['Subject'].fillna('') + ' ' + raw_data['Message'].fillna(''),
    'label': (raw_data['Spam/Ham'] == 'spam').astype(int)
})

df['text'] = df['text'].str.strip()
df = df[df['text'].str.len() > 0].drop_duplicates(subset='text')

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Train samples: 24369
Test samples: 6093


In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

enron_words = {'enron', 'vince', 'kaminski', 'ect', 'hou'}
custom_stop_words = list(ENGLISH_STOP_WORDS.union(enron_words))

model = Pipeline([
    ('vect', CountVectorizer(stop_words=custom_stop_words, min_df=3, max_df=0.9)),
    ('clf', MultinomialNB(alpha=1.0))
])

In [8]:
from sklearn.metrics import classification_report

print("Training model...")
model.fit(X_train, y_train)

print("Evaluating model...")
y_pred = model.predict(X_test)

print("\n" + "=" * 55)
print("Classification Report (Exercise 1: Custom Stopwords)")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam'], digits=4))

Training model...
Evaluating model...

Classification Report (Exercise 1: Custom Stopwords)
              precision    recall  f1-score   support

         Ham     0.9855    0.9840    0.9847      3182
        Spam     0.9825    0.9842    0.9834      2911

    accuracy                         0.9841      6093
   macro avg     0.9840    0.9841    0.9840      6093
weighted avg     0.9841    0.9841    0.9841      6093

